# CECS 478 Course Project: Privacy Review of a Campus Data Release

Team number: Group 6

Members: Adam Nguyen, Sothyra Yo, Leslie Lopez Arjon


Scenario:
The Harbor Area Food Alliance, a fictional nonprofit, wants the responses to the university's fall basic-
needs survey (3,619 respondents). It will use them to choose food-pantry locations and to support grant
applications. The Alliance also plans to post the dataset on its website with its report so other nonprofits
can use it. The survey consent form told students that their answers would be reported only in summary
form.
The Alliance needs food-insecurity rates by housing, work hours, first-generation status, and transfer
status, food-insecurity rates by home ZIP area, and the distribution of PHQ-2 depression-screening scores
among food-insecure students.

In [16]:
from pathlib import Path
import pandas as pd

# Set to "A_Health_Center", "B_Learning_Platform", or "C_Basic_Needs_Survey".
SCENARIO = "C_Basic_Needs_Survey"
DATA = Path(SCENARIO)

text_fields = {"home_zip": str, "student_id": str}
release = pd.read_csv(DATA / "release_request.csv", dtype=text_fields)
auxiliary = pd.read_csv(DATA / "public_auxiliary.csv", dtype=text_fields | {"birth_year": "Int64"})
roster = pd.read_csv(DATA / "leaked_roster.csv", dtype=text_fields)

print("release_request.csv:", release.shape)
print("public_auxiliary.csv:", auxiliary.shape)
print("leaked_roster.csv:", roster.shape)

release_request.csv: (3619, 15)
public_auxiliary.csv: (1500, 8)
leaked_roster.csv: (250, 3)


In [17]:
release.head()

,student_hash,birth_date,gender,home_zip,major,class_level,residence,transfer,first_generation,international,dependents,work_hours,food_insecure,housing_insecure,phq2_score
0,86193b8a0715d2f5c528413b2bd9056a4c5f6ee7045111...,2001-09-09,Woman,92683,Art,Graduate,Off campus,No,Yes,No,No,11-20,Yes,No,2
1,e42005d1fd5c9ac9c66bb54579b1dc48fb6f4627b24e14...,2006-12-07,Woman,90815,Business Administration,Freshman,Off campus,No,No,No,No,21-30,Yes,No,1
2,670571d2b2e5d674bf14ba4b02543d201832cef1b96d7b...,2001-12-18,Woman,90806,Business Administration,Graduate,Off campus,No,Yes,No,No,0,Yes,Yes,5
3,dd9b1e6791ebf565b648fa2bb51e05a3878ce2aaed0692...,2006-09-01,Woman,92840,Psychology,Sophomore,Off campus,No,Yes,No,No,0,Yes,Yes,3
4,4765d9f8905d17b6495e9ad63d6dd967aab10c71ed967e...,2004-10-20,Man,92703,Business Administration,Senior,Off campus,No,No,No,No,31+,No,No,2


In [18]:
auxiliary.head()

,name,birth_year,gender,home_zip,major,class_level,residence,source
0,Jose Flores,2006,Woman,90620,Kinesiology,Junior,Off campus,Social media profile
1,Julia Gupta,2005,Man,92701,Sociology,Junior,Off campus,People-search site
2,Anthony Lee,1996,Woman,92843,Biology,Senior,Off campus,Social media profile
3,Jacob Thomas,2002,Man,90802,Mechanical Engineering,Graduate,Off campus,Social media profile
4,Vanessa Vu,2006,Woman,90703,Business Administration,Sophomore,Off campus,People-search site


In [19]:
roster.head()

,name,student_id,major
0,Ivan Singh,029566384,Business Administration
1,Jennifer Castillo,029829857,Computer Science
2,Maya Jackson,029443829,English
3,Minh Nguyen,029621140,Business Administration
4,Elena Morales,029798159,Kinesiology


In [20]:
print(release.columns.tolist())

['student_hash', 'birth_date', 'gender', 'home_zip', 'major', 'class_level', 'residence', 'transfer', 'first_generation', 'international', 'dependents', 'work_hours', 'food_insecure', 'housing_insecure', 'phq2_score']


The cell below just checks to make sure we have the correct files and how many rows each csv file have)

In [21]:
print("Scenario:", SCENARIO)
print("Survey rows:", len(release))
print("Auxiliary rows:", len(auxiliary))
print("Roster rows:", len(roster))

Scenario: C_Basic_Needs_Survey
Survey rows: 3619
Auxiliary rows: 1500
Roster rows: 250


## 1. Pseudonym test

In [22]:
import hashlib

def hash_student_id(student_id: str) -> str:
    """Hash a student ID using SHA-256 and return the hexadecimal digest."""
    return hashlib.sha256(student_id.encode()).hexdigest()
roster["test_hash"] = roster["student_id"].apply(hash_student_id)

matches = release.merge(
    roster [["name", "student_id", "test_hash", "major"]],
    left_on = "student_hash",
    right_on = "test_hash",
    how = "inner"
    )

print("Matches found:", len(matches))

Matches found: 69


In the Pseudonym test, it found that there are 69 matches between the leaked roster and the survery dataset. This reveals that the student hashes can be linked to names and student ids when an attacker has access to the leaked roster. Even though the student IDs were hashed, the hashes don't provide enough security because the original IDSs can still be vieewed in the leaked roster and can be hashed using the same method 

## 2. Linkage attack

In [26]:
# This creates a copy of the survey data
survey_link = release.copy()

# this extract birth year from the full birth date
survey_link["birth_year"] = pd.to_datetime(survey_link["birth_date"]).dt.year.astype("Int64")

#this links fields shared by both datasets
link_fields = ["birth_year", "gender", "home_zip", "major", "class_level", "residence"]

# this matches survey records with auxiliary records
linked = survey_link.merge(auxiliary, on=link_fields, how="inner")

#this prints out the total linked records, the unique survery records and unique people linked 
print("Total linked records:", len(linked))
print("Unique survey records linked:", linked["student_hash"].nunique())
print("Unique people linked:", linked["name"].nunique())

Total linked records: 957
Unique survey records linked: 934
Unique people linked: 828


## 3. Defended release

In [27]:
# Make a copy of the original survey
defended = release.copy()

# Remove the student hash
defended = defended.drop(columns=["student_hash"])

# Convert full birth date to birth year
defended["birth_year"] = pd.to_datetime(
    defended["birth_date"]
).dt.year

# Remove the original birth date
defended = defended.drop(columns=["birth_date"])

# Generalize ZIP codes to the first 3 digits
defended["home_zip"] = defended["home_zip"].str[:3] + "**"

# Preview the defended dataset
print("Defended dataset shape:", defended.shape)
display(defended.head())

Defended dataset shape: (3619, 14)


,gender,home_zip,major,class_level,residence,transfer,first_generation,international,dependents,work_hours,food_insecure,housing_insecure,phq2_score,birth_year
0,Woman,926**,Art,Graduate,Off campus,No,Yes,No,No,11-20,Yes,No,2,2001
1,Woman,908**,Business Administration,Freshman,Off campus,No,No,No,No,21-30,Yes,No,1,2006
2,Woman,908**,Business Administration,Graduate,Off campus,No,Yes,No,No,0,Yes,Yes,5,2001
3,Woman,928**,Psychology,Sophomore,Off campus,No,Yes,No,No,0,Yes,Yes,3,2006
4,Man,927**,Business Administration,Senior,Off campus,No,No,No,No,31+,No,No,2,2004


## 4. Attack rerun

In [28]:
# Create copies of the defended and auxiliary datasets
defended_test = defended.copy()
auxiliary_test = auxiliary.copy()

# Generalize the auxiliary ZIP codes the same way
auxiliary_test["home_zip"] = (
    auxiliary_test["home_zip"].str[:3] + "**"
)

# Fields shared by both datasets
link_fields_defended = [
    "birth_year",
    "gender",
    "home_zip",
    "major",
    "class_level",
    "residence"
]

# Repeat the linkage attack
rerun_matches = defended_test.reset_index().merge(
    auxiliary_test,
    on=link_fields_defended,
    how="inner"
)

# Count matching records
print("Total matches after defense:", len(rerun_matches))
print(
    "Unique survey records matched:",
    rerun_matches["index"].nunique()
)
print(
    "Unique people matched:",
    rerun_matches["name"].nunique()
)

Total matches after defense: 1788
Unique survey records matched: 1415
Unique people matched: 944


## 5. Utility

In [29]:
# Compare the number of records
print("Original records:", len(release))
print("Defended records:", len(defended))

# Compare food insecurity percentages
original_food = (
    release["food_insecure"].eq("Yes").mean() * 100
)

defended_food = (
    defended["food_insecure"].eq("Yes").mean() * 100
)

print("Original food insecurity %:", round(original_food, 2))
print("Defended food insecurity %:", round(defended_food, 2))

# Compare food insecurity by ZIP area
original_zip = (
    release.assign(zip_area=release["home_zip"].str[:3])
    .groupby("zip_area")["food_insecure"]
    .apply(lambda x: x.eq("Yes").mean() * 100)
)

defended_zip = (
    defended.assign(zip_area=defended["home_zip"].str[:3])
    .groupby("zip_area")["food_insecure"]
    .apply(lambda x: x.eq("Yes").mean() * 100)
)

utility_comparison = pd.DataFrame({
    "Original %": original_zip,
    "Defended %": defended_zip
})

utility_comparison["Difference"] = (
    utility_comparison["Defended %"]
    - utility_comparison["Original %"]
)

display(utility_comparison)

Original records: 3619
Defended records: 3619
Original food insecurity %: 45.1
Defended food insecurity %: 45.1


,Original %,Defended %,Difference
zip_area,,,
902,44.237918,44.237918,0.0
906,45.000000,45.000000,0.0
907,44.124424,44.124424,0.0
908,46.433991,46.433991,0.0
917,50.000000,50.000000,0.0
921,42.857143,42.857143,0.0
923,41.379310,41.379310,0.0
925,57.142857,57.142857,0.0
926,49.189189,49.189189,0.0


## 6. Differentially private tables

## 7. Output files